In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("./Datasets/flats.csv")

In [ ]:
# removing duplicates on the basis of property id

df_unique = df.drop_duplicates(subset="property_id")

In [ ]:
df_unique.isnull().sum()

In [ ]:
df_unique.columns

In [ ]:
# removing null values from the columns which have less than 2% null values

df_nonan = df_unique.dropna(subset=['property_name', 'link', 'society', 'price', 'area', 'areaWithType','bedRoom', 'bathroom', 'balcony', 'address','floorNum','agePossession','description','property_id'])

In [ ]:
df_nonan.isnull().sum()

In [ ]:
df = df_nonan

Additional room column include wheareas list of classes and it is stored in onject
we extarcted the values from list and did one hot encoding

In [ ]:
df.additionalRoom.fillna("", inplace=True)

In [ ]:
def fn(i):
    return i.split(',')

In [ ]:
# split returns a list of individual rooms from string separaated by commas
df.additionalRoom=df.additionalRoom.apply(fn)

In [ ]:
# transforms each element of a list-like column into a separate row, duplicating the remaining column values accordingly.
df_exp=df.explode(column="additionalRoom")

In [ ]:
df_exp

In [ ]:
# to get one hot encoding

dummies = pd.get_dummies(df_exp.additionalRoom)

In [ ]:
# merge all the rows based on first level of the index

result=dummies.groupby(level=0)

In [ ]:
# sum all the values in each group
result = result.sum()

In [ ]:
result

In [ ]:
result.drop("", axis=1, inplace=True)

In [ ]:
result.Others = result.Others+result.additionalRoom

In [ ]:
result.drop(labels="additionalRoom", inplace=True, axis=1)

In [ ]:
# join to df
df=df.join(result)

In [ ]:
df.drop(labels="additionalRoom", axis=1, inplace=True)

In [ ]:
df = df.drop(labels=286)

price has lac crore converted them to numerical 

In [ ]:
s=set()

for i in df.price.str.split(" "):
    s.add(i[1])
    

In [ ]:
def fun(i):
    s=0
    if i[1]=="Lac":
        s = 100000
    else:
        s=10000000
    
    return float(i[0])*s

In [ ]:
df["Price"]=df.price.str.split(" ").apply(fun)

In [ ]:
df.drop("price", axis=1,inplace=True)

In [ ]:
df.info()

area column has rs per sq/m  changed column name 

In [ ]:
df.drop(labels=["property_id","link"], axis=1, inplace =True)

In [ ]:
df.rename(columns={"area":"price per sq.ft"},inplace=True)

bedroom balcony and bathroom has array(['1 Balcony', '3 Balconies', '2 Balconies', '3+ Balconies',
       'No Balcony'], dtype=object)this type of values converted them to int

In [ ]:
lst =[]
for i in df.bedRoom.str.split(" "):
    lst.append(int((i[0])))
df["Bedroom"]=lst

In [ ]:
lst =[]
for i in df.bathroom.str.split(" "):
    lst.append(int((i[0])))
df["Bathroom"]=lst

In [ ]:
df.balcony.unique()

In [ ]:
lst =[]
for i in df.balcony.str.split(" "):
    x=0
    if(i[0] == "No"): 
        x=0
    elif(i[0] == "3+"): 
        x=3
    else:
        x=int(i[0])
    lst.append(x)
df["Balcony"]=lst

In [ ]:
df.drop(labels=["bedRoom",	"bathroom",	"balcony"],axis=1,inplace=True)

floornum column has   4th   of 4 Floors
         1st   of 3 Floors
       12nd   of 14 Floors
         2nd   of 4 Floors
        5th   of 8 Floors 
 'Basement',
 'Ground',
 'Lower'
type of values and i converted them to two columns total number of floors and floor number

In [ ]:
df.floorNum

In [ ]:
st = []

for i in df.floorNum.str.split(" "):
    st.append(i)
st

we used 1 based indexing for floors 
ground = 1
and basement =0
in dataset we have 1st we converted it to 2 as floor number since 1 is for ground

In [ ]:
lst = []
total=[]

for i in df.floorNum.str.split(" "):
    x=0
    if(i[0] == "Ground"):
        x=1
    elif(i[0] == "Basement" or i[0] == "Lower"):
        x=0
    else:
        x=int("".join(filter(str.isdigit,i[0].replace("\xa0",""))))+1
    lst.append(x)

    total.append(i[-2])
df["FloorNum"]=lst
df["Total floors"] = total


In [ ]:
df.drop("floorNum",axis =1, inplace =True)

In [ ]:
lst=[]
sup=[]

for j in df.areaWithType.str.split(")"):
    for i in j:
        lst.append(i.split("("))
lst

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
# df.price.value_counts().reset_index()
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

In [122]:
import re
df.society=df.society.apply(lambda name: re.sub(r"\d+\.(\d+)?\s?★","",str(name)).strip()).str.lower()

In [ ]:
df.society.value_counts()

society
tulip violet                            75
ss the leaf                             73
shapoorji pallonji joyville gurugram    41
dlf new town heights                    38
shree vardhman victoria                 34
                                        ..
professional society                     1
unitech harmony                          1
.                                        1
emaar mgf palm terraces select           1
spire woods now ananda by alpha corp     1
Name: count, Length: 595, dtype: int64